# 🖐️ MouXe ML - Entrenamiento en Google Colab

Este notebook entrena el modelo BiLSTM para MouXe.

## Pasos:
1. Sube tus datos (data/gestos_raw.npy) a Colab
2. Ejecuta las celdas
3. Descarga el modelo entrenado

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive montado')

In [ ]:
# OPCIÓN 1: Subir archivo directamente desde tu PC
from google.colab import files
print('Sube el archivo gestos_raw.npy cuando se abra el explorador')
uploaded = files.upload()
print('✓ Archivo subido')

In [ ]:
# OPCIÓN 2: Copiar desde Google Drive (comenta la opción 1 si usas esta)
# import shutil
# source_path = '/content/drive/MyDrive/Mouxe/data/gestos_raw.npy'
# dest_path = 'gestos_raw.npy'
# shutil.copy(source_path, dest_path)
# print('✓ Archivo copiado desde Google Drive')

In [ ]:
# Instalar dependencias
!pip install tensorflow numpy mediapipe opencv-python

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dropout, Dense
import os

# Configuración
BUFFER_SIZE = 30
NUM_FEATURES = 63
EPOCHS = 50
BATCH_SIZE = 32

# Cargar datos
data = np.load('gestos_raw.npy', allow_pickle=True).item()

if all(k in data for k in ('X', 'y', 'labels')):
    # Formato ya procesado
    X = data['X']
    y = data['y']
    labels = data['labels']
    print(f"✓ Datos cargados (formato procesado)")
    print(f"  X shape: {X.shape}")
    print(f"  y shape: {y.shape}")
    print(f"  Labels: {labels}")
else:
    # Formato crudo de recolectar.py: {gesto: [secuencia, ...], ...}
    print("✓ Datos cargados (formato crudo) - creando X/y/labels")

    # Asegurar orden consistente de labels
    GESTOS_A_GRABAR = [
        'MOVE', 'LEFT_CLICK', 'RIGHT_CLICK', 'SCROLL',
        'FORWARD', 'BACK', 'PUPPET', 'FIST',
        'PALM', 'ZOOM_IN', 'ZOOM_OUT',
    ]

    labels = GESTOS_A_GRABAR
    X = []
    y = []
    for idx, gesto in enumerate(labels):
        secuencias = data.get(gesto, [])
        for seq in secuencias:
            X.append(seq)
            y.append(idx)

    X = np.array(X, dtype=object)
    y = np.array(y, dtype=np.int32)
    print(f"  Gestos encontrados: {len(labels)}")
    print(f"  Secuencias totales: {len(X)}")

    # Verificar dimensiones (ahora pueden variar)
    min_len = min(len(x) for x in X)
    max_len = max(len(x) for x in X)
    print(f"  Longitud secuencias: min={min_len}, max={max_len}")

    # Padding todas las secuencias a la misma longitud (max)
    X_padded = []
    for seq in X:
        if len(seq) < max_len:
            # Relleno con ceros al final
            padded = np.vstack([seq, np.zeros((max_len - len(seq), NUM_FEATURES), dtype=np.float32)])
            X_padded.append(padded)
        else:
            X_padded.append(seq[:max_len])

    X = np.array(X_padded, dtype=np.float32)
    print(f"  X shape (padded): {X.shape}")

In [ ]:
# Crear modelo BiLSTM
num_classes = len(labels)

model = Sequential([
    tf.keras.layers.Input(shape=(BUFFER_SIZE, NUM_FEATURES)),
    Bidirectional(LSTM(128, return_sequences=True, activation='sigmoid')),
    Dropout(0.1),
    Bidirectional(LSTM(64, return_sequences=True, activation='sigmoid')),
    Dropout(0.1),
    Bidirectional(LSTM(32, return_sequences=False, activation='sigmoid')),
    Dense(128, activation='sigmoid'),
    Dropout(0.1),
    Dense(64, activation='sigmoid'),
    Dropout(0.1),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Convertir labels a one-hot
label_to_idx = {name: idx for idx, (name, _) in enumerate(labels)}
y_numeric = np.array([label_to_idx[y_i] for y_i in y])
y_hot = tf.keras.utils.to_categorical(y_numeric, num_classes=num_classes)

# Entrenar
history = model.fit(
    X, y_hot,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Guardar modelo
os.makedirs('models', exist_ok=True)
model.save('models/gesture_model.h5')
print("✓ Modelo guardado")

# Guardar labels
import pickle
with open('models/gesture_labels.pkl', 'wb') as f:
    pickle.dump(labels, f)
print("✓ Labels guardados")

In [ ]:
# Descargar modelo
from google.colab import files
files.download('models/gesture_model.h5')
files.download('models/gesture_labels.pkl')